### Import relevant libraries

In [ ]:
import os
import json
import io
import asyncio
import gradio as gr
import sendgrid
from typing import List

from pydantic import BaseModel, Field
from duckduckgo_search import DDGS
from sendgrid.helpers.mail import Content, Email, Mail, To
from agents import Agent, Runner, function_tool, trace
from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)

console = Console(force_terminal=True, width=80)
_trace_console: Console | None = None


def _active_console() -> Console:
    return _trace_console if _trace_console is not None else console


### Set up environment

In [ ]:
SENDER_EMAIL = "answers@ailysis.io"

sendgrid_api_key = os.environ.get("SENDGRID_API_KEY")

print("Environment ready.")

### Business Tools

In [ ]:
def _web_search_impl(query: str) -> str:
    """Search the web for market trends and competitor info """
    _active_console().print(f"[bold cyan]Researcher is searching for:[/bold cyan] {query}")
    with DDGS() as ddgs:
        results = [r for r in ddgs.text(query, max_results=3)]
    return json.dumps(results)


def _fetch_subscriber_emails_impl() -> List[str]:
    """Registered subscriber emails."""
    emails = ["xxxxx@gmail.com", "xxxxxx@gmail.com"]
    _active_console().print(
        f"[bold magenta]Database:[/bold magenta] Retrieved {len(emails)} subscribers."
    )
    return emails


def _send_marketing_email_impl(subject: str, html_content: str, recipients: List[str]) -> str:
    """Send via SendGrid."""
    _active_console().print(
        f"[bold green]Dispatching email to {len(recipients)} recipient(s)...[/bold green]"
    )
    api_key = os.environ.get("SENDGRID_API_KEY")
    if not api_key:
        return "Skipped send: SENDGRID_API_KEY is not set (dry run)."
    try:
        sg = sendgrid.SendGridAPIClient(api_key=sendgrid_api_key)
        from_email = Email(SENDER_EMAIL)
        for email_addr in recipients:
            to_email = To(email_addr)
            html = Content("text/html", html_content)
            mail = Mail(from_email, to_email, subject, html).get()
            response = sg.client.mail.send.post(request_body=mail)
            if response.status_code not in (200, 201, 202):
                return f"SendGrid error {response.status_code}: {response.body}"
        return f"Successfully sent to {len(recipients)} recipients."
    except Exception as e:
        return f"Error: {str(e)}"


@function_tool
def web_search(query: str) -> str:
    """Search the web for market trends and competitor info."""
    return _web_search_impl(query)


@function_tool
def fetch_subscriber_emails() -> List[str]:
    """Queries the database for registered user emails."""
    return _fetch_subscriber_emails_impl()


@function_tool
def send_marketing_email(subject: str, html_content: str, recipients: List[str]) -> str:
    """Sends the finalized marketing email via SendGrid."""
    return _send_marketing_email_impl(subject, html_content, recipients)

### Agents and orchestrated pipeline


In [ ]:
HOW_MANY_SEARCHES = 3


class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search matters for the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="Web searches to perform to best answer the marketing research query."
    )


class EmailDraft(BaseModel):
    subject: str = Field(description="Catchy subject line.")
    html_body: str = Field(description="The marketing email in HTML format.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=(
        f"You are a research assistant. Given a marketing campaign brief, propose "
        f"{HOW_MANY_SEARCHES} web searches to gather trends, competitors, and positioning. "
        f"Output exactly {HOW_MANY_SEARCHES} search items."
    ),
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

researcher = Agent(
    name="Researcher",
    instructions=(
        "Find market trends and competitor positioning for the campaign. "
        "Call web_search at least once, then synthesize concise findings in your final message."
    ),
    tools=[web_search],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

copywriter = Agent(
    name="Copywriter",
    instructions=(
        "Turn the research into one high-converting HTML marketing email. "
        "Output must match the EmailDraft schema (subject + html_body)."
    ),
    output_type=EmailDraft,
    model="gpt-4o-mini",
)


async def plan_searches(query: str) -> WebSearchPlan:
    result = await Runner.run(planner_agent, f"Query: {query}")
    return result.final_output


async def search_item(item: WebSearchItem) -> str:
    prompt = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(researcher, prompt, max_turns=15)
    return str(result.final_output)


async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    tasks = [asyncio.create_task(search_item(item)) for item in search_plan.searches]
    return list(await asyncio.gather(*tasks))


async def write_email_draft(query: str, search_results: list[str]) -> EmailDraft | None:
    payload = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(copywriter, payload, max_turns=20)
    out = result.final_output
    return out if isinstance(out, EmailDraft) else None

### Streaming logic

In [ ]:
async def run_automation_stream(user_prompt: str):
    """Gradio stream: (plan → search → write → send)."""
    global _trace_console
    buf = io.StringIO()
    capture_console = Console(file=buf, force_terminal=True, width=100)
    _trace_console = capture_console

    def emit() -> str:
        return f"```ansi\n{buf.getvalue()}\n```"

    try:
        capture_console.print("[bold bright_cyan]Starting research...[/bold bright_cyan]")
        yield emit()

        with trace("Campaign Trace"):
            capture_console.print("[yellow]Planning searches...[/yellow]")
            yield emit()
            search_plan = await plan_searches(user_prompt)
            capture_console.print(
                f"[bold green]Will perform {len(search_plan.searches)} searches[/bold green]"
            )
            yield emit()

            capture_console.print("[cyan]Searching...[/cyan]")
            yield emit()
            search_results = await perform_searches(search_plan)
            capture_console.print("[bold green]Finished searching[/bold green]")
            yield emit()

            capture_console.print("[magenta]Thinking about report...[/magenta]")
            yield emit()
            draft = await write_email_draft(user_prompt, search_results)
            capture_console.print("[bold green]Finished writing report[/bold green]")
            yield emit()

            if draft is None:
                capture_console.print("[bold red]Copy step failed; send skipped.[/bold red]")
                yield emit()
                return

            recipients = _fetch_subscriber_emails_impl()
            capture_console.print("[yellow]Writing email...[/yellow]")
            yield emit()
            send_status = _send_marketing_email_impl(
                draft.subject, draft.html_body, recipients
            )
            capture_console.print(f"[dim]{send_status}[/dim]")
            if send_status.startswith("Successfully"):
                capture_console.print("[bold green]Email sent[/bold green]")
            else:
                capture_console.print("[bold red]Email was not sent — see status above.[/bold red]")
            yield emit()

        capture_console.print("[bold bright_yellow]Hooray![/bold bright_yellow]")
        yield emit()

    except Exception as e:
        capture_console.print(f"\n[bold red]Critical error:[/bold red] {e!s}")
        yield emit()
    finally:
        _trace_console = None

### UI with execution trace

In [ ]:
def launch_ui():
    # Use a dark theme 
    with gr.Blocks(theme=gr.themes.Default(), title="AI Marketing Service") as demo:
        gr.Markdown("# Agentic Email Marketing Service")
        
        with gr.Row():
            with gr.Column(scale=1):
                prompt_input = gr.Textbox(
                    label="Campaign Objective", 
                    placeholder="e.g., Promote our new eco-friendly sneakers...",
                    lines=5
                )
                launch_btn = gr.Button("Execute Full Flow", variant="primary")
            
            with gr.Column(scale=2):
                trace_output = gr.Markdown(
                    label="Real-time Execution Trace",
                    sanitize_html=False,
                )

        launch_btn.click(
            fn=run_automation_stream,
            inputs=prompt_input,
            outputs=trace_output,
            show_progress="minimal",
        )

    demo.launch(share=True)

if __name__ == "__main__":
    launch_ui()